# Segunda fase

In [7]:
import pandas

# Importing classes of the project

# Reload classes in memory every time this code block is executed
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from model_building.KNN import KNN
from data_analysis.analizer import DataAnalizer


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
df = pd.read_csv("../out/dataset.csv")
del df["congestion_surcharge"]
del df["pickup_time_in_seconds"]
del df["dropoff_time_in_seconds"]

# Remove all rows with any missing values (NaN)
df= df.dropna()
df


,trip_distance,fare_amount,tip_amount,tolls_amount,extra,passenger_count,pickup_hour,pickup_day_of_week,pickup_day_of_month,pickup_month,dropoff_hour,dropoff_day_of_week,dropoff_day_of_month,dropoff_month,mta_tax,vendorid,ratecodeid,pulocationid,dolocationid,payment_type
0,0.38,3.5,0.00,0.0,0.5,1.0,0,1,1,1,0,1,1,1,0.5,2.0,1.0,170.0,170.0,2.0
1,1.40,6.5,4.00,0.0,0.5,1.0,0,1,1,1,0,1,1,1,0.5,1.0,1.0,229.0,141.0,1.0
2,1.20,7.0,1.70,0.0,0.5,1.0,0,1,1,1,0,1,1,1,0.5,1.0,1.0,144.0,158.0,1.0
3,2.39,10.0,0.00,0.0,0.5,1.0,0,1,1,1,0,1,1,1,0.5,2.0,1.0,244.0,69.0,2.0
4,9.44,28.0,5.00,0.0,0.5,1.0,0,1,1,1,1,1,1,1,0.5,2.0,1.0,114.0,42.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84564,1.22,7.0,3.24,0.0,0.5,1.0,23,1,31,12,23,1,31,12,0.5,2.0,1.0,90.0,68.0,1.0
84565,0.40,4.0,1.55,0.0,3.0,2.0,23,1,31,12,23,1,31,12,0.5,1.0,1.0,79.0,107.0,1.0
84566,1.48,7.5,3.39,0.0,0.5,1.0,23,1,31,12,23,1,31,12,0.5,2.0,1.0,161.0,234.0,1.0
84567,0.90,5.5,0.00,0.0,0.5,5.0,23,1,31,12,22,2,1,1,0.5,2.0,1.0,68.0,246.0,2.0


In [9]:
from sklearn.preprocessing import StandardScaler



# =============================================
# 1. REGRESSION DATASET (continuous target)
# =============================================

# Separate features and target
X_reg = df.drop(columns=['fare_amount'])
y_reg = df['fare_amount']

# Scale only the features (not target)
scaler = StandardScaler()
X_reg_scaled = scaler.fit_transform(X_reg)

# Create scaled DataFrame for regression
df_regression = pd.DataFrame(X_reg_scaled, columns=X_reg.columns)
df_regression['fare_amount'] = y_reg.values  # Add unscaled target

# =============================================
# 2. CLASSIFICATION DATASET (categorical target)
# =============================================

# Create fare classes
bins = [-np.inf, 10, 30, 60, np.inf]
labels = [1, 2, 3, 4]

# Create classification target
df_classification = df.copy()
df_classification['fare_class'] = pd.cut(
    df['fare_amount'],
    bins=bins,
    labels=labels
)

# Separate features and target
X_clf = df_classification.drop(columns=['fare_amount', 'fare_class'])
y_clf = df_classification['fare_class']

# Scale features using SAME scaler (important for consistency)
X_clf_scaled = scaler.transform(X_clf)  # Use existing scaler

# Create scaled DataFrame for classification
df_classification_scaled = pd.DataFrame(X_clf_scaled, columns=X_clf.columns)
df_classification_scaled['fare_class'] = y_clf.values  # Add target

# =============================================
# Verification
# =============================================
print("Regression dataset:")
print(df_regression.head())

print("\nClassification dataset:")
print(df_classification_scaled.head())

print(df_classification['fare_class'].value_counts(normalize=True))

Regression dataset:
   trip_distance  tip_amount  tolls_amount     extra  passenger_count  \
0      -0.666634   -0.760919     -0.229742 -0.463238        -0.470065   
1      -0.404075    0.619532     -0.229742 -0.463238        -0.470065   
2      -0.455557   -0.174228     -0.229742 -0.463238        -0.470065   
3      -0.149237   -0.760919     -0.229742 -0.463238        -0.470065   
4       1.665513    0.964644     -0.229742 -0.463238        -0.470065   

   pickup_hour  pickup_day_of_week  pickup_day_of_month  pickup_month  \
0    -2.316139           -1.018927            -1.674574     -1.532716   
1    -2.316139           -1.018927            -1.674574     -1.532716   
2    -2.316139           -1.018927            -1.674574     -1.532716   
3    -2.316139           -1.018927            -1.674574     -1.532716   
4    -2.316139           -1.018927            -1.674574     -1.532716   

   dropoff_hour  dropoff_day_of_week  dropoff_day_of_month  dropoff_month  \
0     -2.286150          

In [10]:
# Regression Analysis of KNN
analizer_reg = DataAnalizer(df_regression, "fare_amount", test_size=0.2)

rmse_results = {}
for i in range(5, 26, 5):
    print(f"Training KNN Regression for k = {i}")
    knn_reg = KNN(i, problem_type="regression")
    knn_reg.fit(analizer_reg.data_train, analizer_reg.labels_train)
    pred_values = knn_reg.predict(analizer_reg.data_test)

    # Calculate Root Mean Squared Error
    rmse = np.sqrt(np.mean((pred_values - analizer_reg.labels_test)**2))
    rmse_results[i] = rmse
    print(f"k={i}: RMSE = {rmse:.2f}")

print("\nFinal Regression Results:")
print(rmse_results)

Data divided successfully.
Training KNN Regression for k = 5


KeyboardInterrupt: 

In [11]:
# Classification Analysis of KNN
analizer_clf = DataAnalizer(df_classification_scaled, "fare_class", test_size=0.2)

precision_results = {}
for i in range(5, 26, 5):
    print(f"Training KNN Classification for k = {i}")
    knn_clf = KNN(i, problem_type="classification")
    knn_clf.fit(analizer_clf.data_train, analizer_clf.labels_train)
    pred_labels = knn_clf.predict(analizer_clf.data_test)

    # Calculate Precision (accuracy)
    precision = np.mean(pred_labels == analizer_clf.labels_test)
    precision_results[i] = precision
    print(f"k={i}: Precision = {precision:.4f}")

print("\nFinal Classification Results:")
print(precision_results)

Data divided successfully.
Training KNN Classification for k = 5


KeyboardInterrupt: 